# exp14 — structure viewer

Every prediction this experiment produced, for every predictor, next to the
ground truth.

**What is here.** MarinFold [exp245](https://github.com/Open-Athena/MarinFold/tree/main/experiments/exp245_evals_foldbench_held_out_monomers)'s
held-out FoldBench monomer sets, folded by Helico under several contact
conditions and by four baselines:

| predictor | what it saw |
| --- | --- |
| `off` | Helico, one sequence, no contacts |
| `mf_L`, `mf_L2`, `mf_L5` | Helico + MarinFold contacts, top-L / L/2 / L/5 |
| `v2ss`, `v2msa` | Helico + contacts read off a Protenix-v2 structure |
| `oracle` | Helico + ground-truth contacts (the ceiling) |
| `protenix_v2_single_seq`, `protenix_v2_msa` | Protenix-v2 itself |
| `esmfold`, `esmfold2` | ESMFold and ESMFold2, single sequence |

Helico is MSA-free throughout. Nothing here needs a GPU — everything is
downloaded from a public HuggingFace bucket.

Source: [Open-Athena/helico#14](https://github.com/Open-Athena/helico/issues/14).


## Setup

Installs, then pulls the scores and the structures. A few hundred MB, a minute or two.

In [ ]:
# gemmi >= 0.7 is not optional: 0.6.x parses ESMFold2's minimal mmCIF
# to zero models, which shows up as a structure that silently will not draw.
!pip install -q py3Dmol 'gemmi>=0.7' pandas 'huggingface_hub>=1.5'

In [ ]:
import gzip, tarfile
from pathlib import Path

import gemmi
import numpy as np
import pandas as pd
import py3Dmol
from huggingface_hub import HfApi

BUCKET = "timodonnell/helico-experiments"
PREFIX = "exp14_foldbench_held_out_monomers"
LOCAL = Path("exp14"); LOCAL.mkdir(exist_ok=True)
# token=False: the bucket is public, and Colab sometimes has a stale token
# lying around that would otherwise be sent and rejected.
API = HfApi(token=False)


def fetch(relpath: str) -> Path:
    """Download one object from the public bucket, once."""
    out = LOCAL / relpath
    if out.exists():
        return out
    out.parent.mkdir(parents=True, exist_ok=True)
    API.download_bucket_files(BUCKET, [(f"{PREFIX}/{relpath}", str(out))])
    return out


def fetch_structures(relpath: str) -> Path:
    """Download and unpack one structure tarball, once."""
    stem = Path(relpath).name.replace(".tar.gz", "")
    target = LOCAL / "unpacked" / stem
    if target.exists():
        return target
    archive = fetch(relpath)
    target.mkdir(parents=True, exist_ok=True)
    with tarfile.open(archive) as tar:
        tar.extractall(target)
    return target


## The scoreboard

One row per protein, one column per predictor. Sort it, filter it, pick a target to look at.

In [ ]:
per_target = pd.read_csv(fetch("scores/per_target.csv"))
targets = pd.read_csv(fetch("targets.csv"))

METRIC = "lddt"   # or "tm_score", "gdt_ts", "rmsd"

scores = (per_target[per_target.status == "ok"]
          .pivot_table(index="target_id", columns="arm", values=METRIC))
meta = targets.set_index("target_id")[
    ["eval_set", "L_helico", "is_viral", "designed", "exp199_stratum"]]
table = meta.join(scores).rename(columns={"L_helico": "n_residues"})
print(f"{len(table)} proteins x {scores.shape[1]} predictors, metric = {METRIC}")
table.head(20)

In [ ]:
# Where does MarinFold conditioning help most, and least?
cols = [c for c in ("off", "mf_L", "oracle", "protenix_v2_single_seq",
                    "protenix_v2_msa", "esmfold", "esmfold2") if c in table]
view = table[["eval_set", "n_residues"] + cols].copy()
view["mf_L - off"] = view["mf_L"] - view["off"]
view.sort_values("mf_L - off", ascending=False).head(10).round(3)

## Look at a structure

`show(pdb_id)` renders the ground truth in grey with each prediction
superimposed on it. Superposition is a Kabsch fit on the CA atoms the two
structures share — the same atoms the TM-score and RMSD in the table are
computed over — so what you see is what was measured.

Pass `superimpose=False` to draw the models in their original frames instead.

In [ ]:
ARCHIVES = {
    "off": "structures/helico/off.tar.gz",
    "mf_L": "structures/helico/mf_L.tar.gz",
    "mf_L2": "structures/helico/mf_L2.tar.gz",
    "mf_L5": "structures/helico/mf_L5.tar.gz",
    "v2ss": "structures/helico/v2ss.tar.gz",
    "v2msa": "structures/helico/v2msa.tar.gz",
    "oracle": "structures/helico/oracle.tar.gz",
    "protenix_v2_single_seq": "structures/protenix_v2/single_seq.tar.gz",
    "protenix_v2_msa": "structures/protenix_v2/msa.tar.gz",
    "esmfold": "structures/esmfold/esmfold.tar.gz",
    "esmfold2": "structures/esmfold2/esmfold2.tar.gz",
    "ground_truth": "structures/ground_truth.tar.gz",
}

COLORS = {"ground_truth": "#9a9a9a", "off": "#5c5c5c", "mf_L": "#b8452f",
          "mf_L2": "#cd6b53", "mf_L5": "#e0937f", "v2ss": "#7a4fbf",
          "v2msa": "#1a7f5a", "oracle": "#1b5e9c",
          "protenix_v2_single_seq": "#a0762b", "protenix_v2_msa": "#2e7d32",
          "esmfold": "#c74a86", "esmfold2": "#7a6a3a"}


def structure_path(arm: str, pdb_id: str) -> Path | None:
    """The file holding `arm`'s model of `pdb_id`, downloading if needed."""
    root = fetch_structures(ARCHIVES[arm])
    for pattern in (f"{pdb_id}.pdb.gz", f"{pdb_id}.cif.gz", f"{pdb_id}.cif",
                    f"{pdb_id}.pdb"):
        hits = sorted(root.rglob(pattern))
        if hits:
            return hits[0]
    # Protenix and ESMFold keep a directory per target.
    for pattern in (f"**/{pdb_id}/**/*_sample_0.cif", f"**/{pdb_id}/structure.cif"):
        hits = sorted(root.rglob(pattern))
        if hits:
            return hits[0]
    return None


def read_structure(path: Path) -> gemmi.Structure:
    """Parse a prediction or ground truth, whatever format it arrived in.

    Everything goes through `gemmi.read_structure` on a real file rather than
    `make_structure_from_block`: the latter returns a structure with zero
    models for ESMFold2's mmCIF, which then fails far away from the cause.
    """
    data = path.read_bytes()
    if path.suffix == ".gz":
        data = gzip.decompress(data)
        suffix = Path(path.stem).suffix
    else:
        suffix = path.suffix
    tmp = LOCAL / "_scratch" / f"read{suffix}"
    tmp.parent.mkdir(parents=True, exist_ok=True)
    tmp.write_bytes(data)
    structure = gemmi.read_structure(str(tmp))
    structure.setup_entities()
    if not len(structure):
        raise ValueError(f"{path} parsed to zero models")
    return structure


def ca_atoms(structure: gemmi.Structure) -> dict[int, np.ndarray]:
    out = {}
    for chain in structure[0]:
        for i, residue in enumerate(chain):
            atom = residue.find_atom("CA", "*")
            if atom is not None:
                out.setdefault(i, np.array([atom.pos.x, atom.pos.y, atom.pos.z]))
    return out


def kabsch(mobile: np.ndarray, target: np.ndarray):
    """Rotation and translation putting `mobile` onto `target`."""
    mc, tc = mobile.mean(axis=0), target.mean(axis=0)
    u, _, vt = np.linalg.svd((mobile - mc).T @ (target - tc))
    d = np.sign(np.linalg.det(vt.T @ u.T))
    rotation = vt.T @ np.diag([1.0, 1.0, d]) @ u.T
    return rotation, tc - rotation @ mc


def superimposed_pdb(model_path: Path, gt_path: Path) -> str:
    """`model_path` as a PDB string, Kabsch-fitted onto the ground truth.

    Fitted on the CA atoms shared by residue index, which is the correspondence
    the reported TM-score and RMSD use. Falls back to the unfitted model when
    fewer than three residues line up.
    """
    model, truth = read_structure(model_path), read_structure(gt_path)
    m_ca, t_ca = ca_atoms(model), ca_atoms(truth)
    shared = sorted(set(m_ca) & set(t_ca))
    if len(shared) >= 3:
        rotation, translation = kabsch(
            np.stack([m_ca[i] for i in shared]),
            np.stack([t_ca[i] for i in shared]))
        for chain in model[0]:
            for residue in chain:
                for atom in residue:
                    v = rotation @ np.array([atom.pos.x, atom.pos.y, atom.pos.z]) + translation
                    atom.pos = gemmi.Position(*v)
    model.setup_entities()
    return model.make_pdb_string()


def show(pdb_id: str, arms=("mf_L", "oracle"), superimpose=True,
         show_ground_truth=True, width=900, height=600, style="cartoon"):
    """Render one protein's predictions, optionally fitted onto the truth."""
    gt_path = structure_path("ground_truth", pdb_id)
    if gt_path is None:
        raise FileNotFoundError(f"no ground truth for {pdb_id}")

    viewer = py3Dmol.view(width=width, height=height)
    if show_ground_truth:
        viewer.addModel(read_structure(gt_path).make_pdb_string(), "pdb")
        viewer.setStyle({"model": -1},
                        {style: {"color": COLORS["ground_truth"], "opacity": 0.65}})
    for arm in arms:
        path = structure_path(arm, pdb_id)
        if path is None:
            print(f"[skip] {arm}: no structure for {pdb_id}")
            continue
        try:
            text = (superimposed_pdb(path, gt_path) if superimpose
                    else read_structure(path).make_pdb_string())
        except Exception as error:  # one unreadable file must not lose the rest
            print(f"[skip] {arm}: {type(error).__name__}: {error}")
            continue
        viewer.addModel(text, "pdb")
        viewer.setStyle({"model": -1}, {style: {"color": COLORS.get(arm, "#333333")}})
    viewer.zoomTo()

    legend = [("ground truth", COLORS["ground_truth"])] if show_ground_truth else []
    legend += [(a, COLORS.get(a, "#333333")) for a in arms]
    print(pdb_id + "   " + "   ".join(f"{name} ({color})" for name, color in legend))
    row = table.loc[pdb_id] if pdb_id in table.index else None
    if row is not None:
        shown = [a for a in arms if a in table.columns]
        print(f"{METRIC}: " + "  ".join(f"{a}={row[a]:.3f}" for a in shown
                                        if pd.notna(row[a])))
    return viewer.show()

### A protein where contacts help a lot

Grey is the truth; the coloured models are fitted onto it.

In [ ]:
best = (table.assign(gain=table["mf_L"] - table["off"])
              .sort_values("gain", ascending=False).index[0])
show(best, arms=("off", "mf_L", "oracle"))

### The same protein against the baselines

In [ ]:
show(best, arms=("protenix_v2_single_seq", "protenix_v2_msa",
                 "esmfold", "esmfold2"))

### Pick your own

Any id in `table.index`. Set `superimpose=False` to see the raw frames.

In [ ]:
show("8oqh_A", arms=("mf_L", "oracle", "esmfold2"), superimpose=True)